In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import  ChatOpenAI
from langchain.tools import tool

from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results=3,
    search_depth="advanced",
    include_answer=True,
)


llm = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
@tool
def fetch_live_external_data(topic: str) -> str:
    """Fetch live or real-time external data for a topic (e.g. weather, stock quote, breaking news).

    Use this when the user asks for up-to-the-minute or live information.
    """
    # Demo only: always fail so wrap_tool_call can demonstrate exception handling
    raise ConnectionError("Simulated failure: external data service is unreachable.")

In [ ]:
from langchain.agents.middleware import  wrap_model_call,wrap_tool_call,ModelRequest

@wrap_model_call
def logging_middleware(request:ModelRequest,handler):

    messages = request.state["messages"]

    last_message = messages[-1] 

    print(f"Logging Middleware: Last message role: {last_message}")

    response= handler(request)

    

    print(f"Logging Middleware: Model response: {response}")

    return response

In [ ]:

@wrap_model_call
def dynamic_model_middleware(request:ModelRequest,handler):

    messages = request.state["messages"]

    last_message = messages[-1] 

    complex_keywords = ["complex", "multiple", "detailed", "analysis", "explain"]

    if last_message and last_message.content:
        last_message_content = last_message.content.lower()

        if any(keyword in last_message_content for keyword in complex_keywords):
            print("Dynamic Model Middleware: Using gpt-4.1 for complex query.")
            #request.model = ChatOpenAI(model="gpt-4.1")
            request.override(model=ChatOpenAI(model="gpt-4.1"))
           
        else:
            print("Dynamic Model Middleware: Using gpt-4.1-mini for simple query.")
            #request.model = ChatOpenAI(model="gpt-4.1-mini")
            request.override(model=ChatOpenAI(model="gpt-4.1-mini"))

    response= handler(request)



    return response

In [ ]:
from langchain.messages import  ToolMessage
from langchain.agents.middleware import  wrap_tool_call,ModelRequest,ToolCallRequest


@wrap_tool_call
def handleToolErrorMiddleware(request:ToolCallRequest,handler):

    try:
        response= handler(request)
        return response
    except Exception as e:
        print(f"Tool Error Middleware: Caught exception: {e}")
        # Handle the error gracefully, e.g., return a default response or log the error
        return ToolMessage(content=f"Error occurred while calling the tool: {e}. Give a simulated response instead.", 
                           tool_call_id=request.tool_call["id"]) 

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[ fetch_live_external_data],
    middleware=[logging_middleware,dynamic_model_middleware,handleToolErrorMiddleware]
)


NameError: name 'fetch_live_external_data' is not defined

In [ ]:
response= agent.invoke(
    {
        "messages":[
            {"role": "user", "content": " what is the latest news on AI research and development?"}
        ]
    }
)

response

In [ ]:
from langchain.agents.middleware import  PIIMiddleware,SummarizationMiddleware
from langgraph.checkpoint.memory import  InMemorySaver

agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model= "gpt-4.1-mini",
            trigger=[("messages",6),("tokens",1000),("fraction",0.5)],
            keep=("messages",2),
            
        )

        ,PIIMiddleware(
            pii_type="email",
            strategy="block",  # Options: "block", "mask", "replace"
            apply_to_input=True,
            apply_to_output=True
            
        ),
        PIIMiddleware(
            pii_type="api_key",
            strategy="mask",  # Options: "block", "mask", "replace"
            detector=r"[a-zA-Z0-9]{20,}",  # Example regex for API keys
            apply_to_input=True,
        )
    ]
)



In [5]:
config = {"configurable": {"thread_id": "1"}}

result= agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "i Love mumbai"}
        ]
    }, config=config
)
result

{'messages': [HumanMessage(content='i Love mumbai', additional_kwargs={}, response_metadata={}, id='f0266019-f59e-42f7-b444-42af4c1fe365'),
  AIMessage(content="That's great to hear! Mumbai is a vibrant and diverse city known for its culture, food, and bustling energy. What do you love most about Mumbai?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 11, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_959487c384', 'id': 'chatcmpl-E43CPNs6ypLS36y5rHAJLTnnAjSuA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f847e-8de4-7860-ba92-6f376522c2fd-0', tool_calls=[], invalid_tool_calls=[], us

In [6]:
config = {"configurable": {"thread_id": "1"}}

result= agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "i Love python"}
        ]
    }, config=config
)
result

{'messages': [HumanMessage(content='i Love mumbai', additional_kwargs={}, response_metadata={}, id='f0266019-f59e-42f7-b444-42af4c1fe365'),
  AIMessage(content="That's great to hear! Mumbai is a vibrant and diverse city known for its culture, food, and bustling energy. What do you love most about Mumbai?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 11, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_959487c384', 'id': 'chatcmpl-E43CPNs6ypLS36y5rHAJLTnnAjSuA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f847e-8de4-7860-ba92-6f376522c2fd-0', tool_calls=[], invalid_tool_calls=[], us

In [7]:
config = {"configurable": {"thread_id": "1"}}

result= agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "i Love Java"}
        ]
    }, config=config
)
result

{'messages': [HumanMessage(content='i Love mumbai', additional_kwargs={}, response_metadata={}, id='f0266019-f59e-42f7-b444-42af4c1fe365'),
  AIMessage(content="That's great to hear! Mumbai is a vibrant and diverse city known for its culture, food, and bustling energy. What do you love most about Mumbai?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 11, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_959487c384', 'id': 'chatcmpl-E43CPNs6ypLS36y5rHAJLTnnAjSuA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f847e-8de4-7860-ba92-6f376522c2fd-0', tool_calls=[], invalid_tool_calls=[], us

In [8]:
config = {"configurable": {"thread_id": "1"}}

result= agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "I love vada pav"}
        ]
    }, config=config
)
result

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user expresses enthusiasm and positive feelings toward Mumbai, Python, and Java. The overall task is to acknowledge and engage with the user's interests in these topics.\n\n## SUMMARY\n\n- The user stated affection for Mumbai, Python, and Java.\n- The AI responded by affirming the qualities of Mumbai as a vibrant city and the versatility of Python as a programming language.\n- The AI sought to engage further by asking what the user loves most about Mumbai and Python and whether they are working on any Python projects.\n- The user has not yet provided details about projects or specific reasons for their preferences.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\n- Await user responses elaborating on their interests in Mumbai, Python, or Java.\n- Engage with the user by providing relevant information, assistance, or discussions related to these topics.", additional_kwargs={'lc_sour

In [36]:
from IPython.display import display, Markdown

Markdown(result['messages'][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user is expressing their interests, specifically their fondness for Mumbai, Python, and Java. The underlying intent appears to be sharing preferences or possibly seeking engagement related to these topics.

## SUMMARY

- The user stated that they love Mumbai.
- They also expressed love for Python and Java programming languages.
- The AI responded with positive affirmations and offered to assist with Python-related topics or projects.
- There was no explicit request or detailed question posed by the user beyond these expressions.

## ARTIFACTS

None

## NEXT STEPS

- Clarify the user's intent regarding Python and Java to determine if they need assistance, resources, or project ideas.
- Engage the user by asking what specifically they would like help with related to Mumbai, Python, or Java.